# 00 — Prueba de concepto del clasificador (funcionalidad 010, CA1)

Recorre **todo el pipeline** con pocas imágenes (10 000 como máximo) para validar el diseño y medir tiempos antes del entrenamiento completo. **No se exige umbral de desempeño.**

```
obtener → validar → preparar → entrenar → exportar → evaluar → publicar (registro local)
```

Este cuaderno **no contiene lógica**: clona el repositorio en un commit fijo, instala el paquete `clasificador` y llama a sus etapas (ADR-0014). Todo lo que hace está probado en la integración continua.

**Requisitos en Kaggle** (*Settings* del cuaderno):

1. *Accelerator*: **GPU T4 x2**. *Internet*: **activado**.
2. *Add-ons → Secrets*, con la casilla tildada para este cuaderno:
   - `HF_TOKEN`: *token* de lectura de Hugging Face.
   - `GITHUB_TOKEN`: *token* de GitHub **de solo lectura** del repositorio (el repositorio es privado).
3. Haber aceptado en Hugging Face los términos de uso de los conjuntos que lo pidan.

**Para ejecuciones largas**: usar *Save Version → Save & Run All (Commit)*. Kaggle ejecuta todo el cuaderno en segundo plano, aunque se cierre el navegador, y guarda la salida y los archivos de `/kaggle/working`. En una sesión interactiva, si queda inactiva, Kaggle la cierra y borra esa carpeta.

Ninguna credencial se imprime ni se guarda en archivos: el token de GitHub viaja en una variable de entorno de git, nunca en la URL, y los errores se muestran sin él. **Si alguna vez un token aparece en pantalla, revocarlo de inmediato.**

## 1. Parámetros de la ejecución

In [ ]:
# Rama o commit del repositorio que se ejecuta. Para resultados reproducibles,
# usar un commit concreto (los 40 caracteres) en lugar de una rama.
REVISION = "claude/intelligent-franklin-yl0b03"
REPOSITORIO = "valentinasalmon/PFG-sistema-deteccion-IA"
VERSION = "0.1.0"  # versión de prueba: no se publica en Hugging Face

from pathlib import Path
TRABAJO = Path("/kaggle/working")
CODIGO = TRABAJO / "codigo"

## 2. Credenciales (desde los secretos de Kaggle)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secretos = UserSecretsClient()
os.environ["HF_TOKEN"] = secretos.get_secret("HF_TOKEN")
_token_github = secretos.get_secret("GITHUB_TOKEN")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
print("Credenciales cargadas (no se muestran).")

## 3. Código: clonar el repositorio e instalar el paquete

In [ ]:
import base64
import subprocess


def git_seguro(*argumentos, token=None):
    """Ejecuta git sin que el token aparezca nunca en el comando ni en los errores.

    El token viaja en una variable de entorno de configuración de git
    (cabecera de autorización), no en la URL. Si git falla, se muestra solo
    su mensaje de error, que no contiene el token.
    """
    entorno = dict(os.environ)
    if token:
        credencial = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        entorno.update({
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credencial}",
            "GIT_TERMINAL_PROMPT": "0",
        })
    proceso = subprocess.run(["git", *argumentos], env=entorno, capture_output=True, text=True)
    if proceso.returncode != 0:
        mensaje = proceso.stderr.replace(token, "***") if token else proceso.stderr
        raise RuntimeError(f"git {argumentos[0]} falló:\n{mensaje.strip()}") from None
    return proceso.stdout.strip()


if CODIGO.exists():
    subprocess.run(["rm", "-rf", str(CODIGO)], check=True)
git_seguro("clone", "--quiet", f"https://github.com/{REPOSITORIO}.git", str(CODIGO),
           token=_token_github)
git_seguro("-C", str(CODIGO), "fetch", "--quiet", "origin", REVISION, token=_token_github)
git_seguro("-C", str(CODIGO), "checkout", "--quiet", REVISION)
del _token_github

commit = git_seguro("-C", str(CODIGO), "rev-parse", "HEAD")
print("Commit en uso:", commit)


In [ ]:
# PyTorch ya viene instalado en Kaggle con soporte de GPU: no se reinstala.
!pip install -q -e "{CODIGO}/red_neuronal[entrenamiento]"
!python -m clasificador --ayuda

## 4. Explorar las columnas de Community Forensics

Los nombres de columnas de la receta (`columna_imagen`, `columna_clase`, `valores_sinteticos`, `columna_generador`) deben coincidir con los del conjunto. Esta celda muestra el esquema y unas filas **sin descargar imágenes**.

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

# Se lee solo el esquema (al final de cada archivo Parquet) y unas filas sin
# la columna de imagen: no se descargan imágenes. Leer "la primera fila" con
# streaming obliga a bajar un bloque entero con cientos de imágenes.
sistema = HfFileSystem(token=os.environ["HF_TOKEN"])
archivos = sorted(sistema.glob("datasets/OwensLab/CommunityForensics-Small/**/*.parquet"))
print(f"{len(archivos)} archivos Parquet. Primeros:")
for archivo in archivos[:3]:
    print("  ", archivo)

with sistema.open(archivos[0], "rb") as abierto:
    parquet = pq.ParquetFile(abierto)
    print("\nEsquema (columnas y tipos):")
    print(parquet.schema_arrow)
    print(f"\nFilas en este archivo: {parquet.metadata.num_rows}")
    livianas = [c.name for c in parquet.schema_arrow
                if not str(c.type).startswith(("binary", "large_binary", "struct"))]
    muestra = parquet.read_row_group(0, columns=livianas).slice(0, 5).to_pandas()
    print("\nPrimeras filas (sin la imagen):")
    print(muestra.to_string())


## 5. Configuración de la prueba de concepto

Se parte del archivo versionado `configuracion/prueba_de_concepto.yaml` y, **si la exploración anterior mostró otros nombres**, se ajustan acá. La configuración efectiva queda guardada en el paquete publicado y en MLflow.

In [ ]:
import yaml

base = yaml.safe_load((CODIGO / "red_neuronal/configuracion/prueba_de_concepto.yaml").read_text())
fuente = base["datos"]["fuentes"][0]

# Ajustar según la salida de la sección 4 (dejar igual si coinciden):
fuente["columna_imagen"] = fuente["columna_imagen"]
fuente["columna_clase"] = fuente["columna_clase"]
fuente["valores_sinteticos"] = fuente["valores_sinteticos"]
fuente["columna_generador"] = fuente["columna_generador"]

base["directorio_trabajo"] = str(TRABAJO / "trabajo")
CONFIGURACION = TRABAJO / "configuracion_efectiva.yaml"
CONFIGURACION.write_text(yaml.safe_dump(base, allow_unicode=True, sort_keys=False))
print(CONFIGURACION.read_text())

## 6. Ejecutar las etapas midiendo el tiempo de cada una

In [ ]:
import json
import shutil
import time

tiempos = {}

def ejecutar_etapa(etapa, *extra):
    """Ejecuta una etapa del pipeline, mide su duración y se detiene si falla."""
    inicio = time.perf_counter()
    proceso = subprocess.run(
        ["python", "-m", "clasificador", etapa, "--configuracion", str(CONFIGURACION), *extra],
        cwd=CODIGO / "red_neuronal", text=True,
    )
    tiempos[etapa] = round(time.perf_counter() - inicio, 1)
    print(f"--> {etapa}: {tiempos[etapa]} s")
    if proceso.returncode != 0:
        raise RuntimeError(f"La etapa '{etapa}' falló: ver el mensaje anterior.")

In [ ]:
ejecutar_etapa("obtener")
ejecutar_etapa("validar")
ejecutar_etapa("preparar")

In [ ]:
ejecutar_etapa("entrenar")
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
ejecutar_etapa("exportar", "--version", VERSION)
ejecutar_etapa("evaluar")

### Publicación en un registro **local**

En la prueba de concepto no se exige el umbral de AUC (CA1): se baja solo para esta ejecución, de modo que se pueda probar la publicación completa. El modelo queda en `/kaggle/working/registro`, **no** en Hugging Face.

In [ ]:
configuracion_publicacion = yaml.safe_load(CONFIGURACION.read_text())
configuracion_publicacion["compuerta"]["auc_minimo"] = 0.51
CONFIGURACION_PUBLICACION = TRABAJO / "configuracion_publicacion_poc.yaml"
CONFIGURACION_PUBLICACION.write_text(yaml.safe_dump(configuracion_publicacion, allow_unicode=True))

inicio = time.perf_counter()
proceso = subprocess.run(
    ["python", "-m", "clasificador", "publicar", "--configuracion", str(CONFIGURACION_PUBLICACION),
     "--version", VERSION, "--registro", f"local:{TRABAJO / 'registro'}"],
    cwd=CODIGO / "red_neuronal", text=True,
)
tiempos["publicar"] = round(time.perf_counter() - inicio, 1)
print("Código de salida:", proceso.returncode)

## 7. Resultados para la bitácora (T24)

In [ ]:
import json
import shutil
from pathlib import Path

# La sección es independiente de las anteriores: si el kernel se reinició,
# igual funciona con lo que las etapas dejaron en el disco.
TRABAJO = Path("/kaggle/working")
VERSION = globals().get("VERSION", "0.1.0")
trabajo = TRABAJO / "trabajo"
necesarios = {
    "entrenamiento": trabajo / "entrenamiento/resultado.json",
    "exportacion": trabajo / "exportacion/exportacion.json",
    "evaluacion": trabajo / "evaluacion/prueba/metricas.json",
}
faltantes = [etapa for etapa, ruta in necesarios.items() if not ruta.exists()]
if faltantes:
    raise RuntimeError(
        f"Faltan los resultados de: {', '.join(faltantes)}. La sesión se cerró y Kaggle "
        "borró /kaggle/working, o esas etapas no terminaron: ejecutar desde la sección 1."
    )

entrenamiento = json.loads(necesarios["entrenamiento"].read_text())
exportacion = json.loads(necesarios["exportacion"].read_text())
metricas = json.loads(necesarios["evaluacion"].read_text())
uso_disco = shutil.disk_usage(TRABAJO)
resumen = {
    "commit": exportacion["commit"],
    "tiempos_segundos": globals().get("tiempos", "se perdieron al reiniciarse el kernel"),
    "mejor_auc_validacion": entrenamiento["mejor_auc"],
    "epoca_mejor": entrenamiento["epoca_mejor"],
    "temperatura": entrenamiento["temperatura"],
    "ece_antes": entrenamiento["ece_antes"],
    "ece_despues": entrenamiento["ece_despues"],
    "auc_prueba": metricas["total"]["auc"],
    "exactitud_prueba": metricas["total"]["exactitud"],
    "latencia_ms_por_imagen_cpu": exportacion["latencia_ms_por_imagen"],
    "diferencia_onnx": exportacion["diferencia_maxima"],
    "disco_usado_gb": round((uso_disco.total - uso_disco.free) / 1e9, 2),
}
(TRABAJO / "resumen_prueba_de_concepto.json").write_text(json.dumps(resumen, indent=2))
print(json.dumps(resumen, indent=2, ensure_ascii=False))


In [ ]:
from IPython.display import Image as MostrarImagen, Markdown, display

for grafico in ("curva_roc.png", "matriz_confusion.png"):
    ruta = trabajo / "evaluacion/prueba" / grafico
    if ruta.exists():
        display(MostrarImagen(str(ruta)))
ficha = TRABAJO / "registro" / f"v{VERSION}" / "ficha.md"
if ficha.exists():
    display(Markdown(ficha.read_text()))
else:
    print("No hay ficha publicada: la etapa 'publicar' no terminó (ver su mensaje en la sección 6).")


## 8. Qué hacer con los resultados

1. Copiar la salida de la sección 7 en el chat o en la entrada de bitácora de la tarea T24.
2. Con los tiempos medidos se estima cuánto tarda el entrenamiento completo (por ejemplo, 50 000 imágenes por clase).
3. Si alguna etapa falló, el mensaje en español indica el motivo: copiarlo tal cual.

**No** se suben al repositorio ni las imágenes ni el modelo de esta carpeta.